# E3: preflight GPU y recibos persistentes en GitHub

Este cuaderno comprueba **las 17 recetas reales con datos artificiales**: 13 rivales neuronales E2, el candidato histórico aditivo de 1210 parámetros y tres comparadores clásicos. Guarda hardware, versiones, fuentes y resultados del contrato. Los neuronales hacen forward, backward y guardado/recarga; ARIMA hace un ajuste sólo sintético. No usa datos de mercado, pesos entrenados, HPO ni métricas de rendimiento.

Abre este cuaderno nuevo; no pegues comandos en la última celda vacía de un cuaderno anterior. En Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU → T4** (o la GPU disponible) y luego **Entorno de ejecución → Ejecutar todas**. El antecedente L4 conserva 17/17 perfiles aprobados según su consola, pero su ZIP se perdió. La nueva ejecución **A100** `20261118T172624131741Z` sí guardó sus recibos en GitHub: [resultados inmutables](https://github.com/jassan-bit/DeepLearning_202630/tree/47d2f316a50d9268cb6484a547e8a2f0cc0413eb/runs/colab_preflight/20261118T172624131741Z), commit `47d2f316a50d9268cb6484a547e8a2f0cc0413eb`, con código `1ff136f71c0ffcd1dbd918efbc078a8d9653ed21`. La auditoría cotejó los 40 miembros del ZIP y los 42 archivos persistidos sin discrepancias de bytes. No reejecutó GPU ni recalculó pesos, tensores o gradientes; el corte CPU y el antecedente L4 conservan alcances distintos.

El cuaderno prepara por defecto **un Python 3.12.12 y Torch 2.8.0+cu126 aislados**, con el lock E2 CUDA completo. El kernel puede ser Python 3.13: el bootstrap obtiene su propio Python 3.12.12. **No cambia el Torch base de Colab ni reinicia el kernel.** La primera descarga de wheels puede sumar varios GB y tardar varios minutos. Tener GPU no acredita novedad, potencia ni R3.


Al terminar, el cuaderno guarda todos los JSON y logs de evidencia, su ZIP y un manifiesto de hashes en una rama nueva de `jassan-bit/DeepLearning_202630`, bajo `runs/colab_preflight/<RUN_ID>/`. La publicación queda identificada por URL y commit; los resultados anteriores conservan sus bytes. La comprobación sintética no habilita la campaña con datos reales.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, zipfile, importlib.util

PREFLIGHT_DEVICE = "cuda:0"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
BASE = Path("/content") if Path("/content").exists() else Path.cwd()
CHECKOUT = BASE / ("e3-preflight-source-" + RUN_ID)
OUTPUT = BASE / ("e3-preflight-evidence-" + RUN_ID)
REPOSITORY = "https://github.com/jassan-bit/DeepLearning_202630.git"
PINNED_SOURCE = globals().get("E3_SOURCE_COMMIT", "main")
if PINNED_SOURCE != "main" and (len(PINNED_SOURCE) != 40 or any(c not in "0123456789abcdef" for c in PINNED_SOURCE)):
    raise ValueError("E3_SOURCE_COMMIT debe ser un SHA completo o main.")

# main aporta la revisión publicada; se guardan el HEAD real y los hashes.
subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout",
                "--branch", "main", REPOSITORY, str(CHECKOUT)], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "sparse-checkout", "init", "--cone"], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "sparse-checkout", "set", "src", "scripts",
                "experiments/e3_remediacion_20261118/evaluation",
                "experiments/e3_ficha_cnn_lstm_atencion_v2_20261116/verification"], check=True)
if PINNED_SOURCE != "main":
    subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", PINNED_SOURCE], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", PINNED_SOURCE], check=True)
HEAD = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
SCRIPT = CHECKOUT / "experiments/e3_remediacion_20261118/evaluation/colab_preflight.py"
assert SCRIPT.is_file(), "La revisión del preflight todavía no está publicada en main."
print("Código consultado:", HEAD)
print("Destino nuevo de evidencia:", OUTPUT)


## Conectar el guardado de GitHub antes de usar la GPU

Colab necesita un token GitHub con **Contents: Read and write**, limitado a `jassan-bit/DeepLearning_202630`. Puedes guardarlo en **Secretos** (icono de llave) con el nombre `GITHUB_TOKEN` y permitir su uso en este cuaderno. Si ya existe se reutiliza; si falta, se pide en una entrada oculta. No lo pegues en el código ni en el chat.

Se comprueba el acceso al repositorio antes de preparar CUDA. Esa consulta de lectura no garantiza el alcance de escritura de un token fino: el permiso Contents debe estar configurado. El token se mantiene en memoria; el entorno de los trabajadores científicos lo excluye.


In [ ]:
import os
from getpass import getpass

def _e3_github_token():
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        try:
            from google.colab import userdata
            token = (userdata.get("GITHUB_TOKEN") or "").strip()
        except Exception:
            token = ""
    if not token:
        token = globals().get("_E3_GITHUB_TOKEN", "").strip()
    if not token:
        token = getpass("GITHUB_TOKEN (Contents: Read and write para este repositorio): ").strip()
    if not token:
        raise RuntimeError("Falta el token; no se inicia la preparación GPU.")
    return token

_E3_GITHUB_TOKEN = _e3_github_token()
github_api_path = SCRIPT.with_name("colab_github.py")
spec = importlib.util.spec_from_file_location("e3_colab_github_storage", github_api_path)
github_api = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = github_api
spec.loader.exec_module(github_api)
try:
    GITHUB_ACCESS = github_api.validate_credentials(_E3_GITHUB_TOKEN)
except Exception:
    _E3_GITHUB_TOKEN = ""
    raise
print("Acceso al repositorio comprobado. La escritura requiere Contents: Read and write.")


## Preparar el runtime CUDA aislado

Se reutilizan el bootstrap y el lock E2 ya publicados, fijados por commit y SHA-256: incluye statsmodels, reformer-pytorch y sus dependencias. La instalación se hace en `/content/e3-preflight-gpu-runtime-v1`. Un entorno completo se verifica y reutiliza; uno parcial se conserva y pide otra carpeta local, por ejemplo `v2`. No modifica entornos ni resultados históricos. Si no hay GPU visible, se detiene antes de descargar wheels CUDA.


In [ ]:
RUNTIME_WORKSPACE = BASE / "e3-preflight-gpu-runtime-v1"
SETUP_ERROR = None
runtime_api_path = SCRIPT.with_name("colab_runtime.py")
spec = importlib.util.spec_from_file_location("e3_isolated_runtime", runtime_api_path)
runtime_api = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime_api)
try:
    GPU_PYTHON, GPU_ENV, runtime_receipt = runtime_api.prepare_runtime(RUNTIME_WORKSPACE)
    print("Intérprete aislado:", GPU_PYTHON)
except Exception as error:
    SETUP_ERROR = type(error).__name__ + ": " + str(error)
    print("Preparación detenida:", SETUP_ERROR)
    print("Se conservan la carpeta y los mensajes de instalación; el kernel base no cambió.")


## Ejecutar los contratos sintéticos

Se descargará únicamente el paquete público de código D02 (7.84 MB), atado a commit, tamaño y SHA-256. Contiene las recetas y 76 archivos de autores con sus licencias. No solicita credenciales ni archivos del estudiante. Cada modelo corre en un proceso aislado. Cualquier fallo deja su recibo y log; no se reemplaza por otra arquitectura.

Las salidas E2 están normalizadas por su escala objetivo; la salida E3 está en unidades originales. Sólo se comprueban forma, finitud, gradiente y recarga; **no se comparan sus números como desempeño**.


In [ ]:
if SETUP_ERROR is None:
    command = [str(GPU_PYTHON), str(SCRIPT), "--device", PREFLIGHT_DEVICE, "--output", str(OUTPUT)]
    CONSOLE_LOG = BASE / ("e3-preflight-console-" + RUN_ID + ".log")
    with CONSOLE_LOG.open("x") as console:
        with subprocess.Popen(command, cwd=CHECKOUT, env=GPU_ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="")
                console.write(line)
                console.flush()
            returncode = process.wait()
    print("Código de salida del preflight:", returncode)
    RECEIPT = OUTPUT / "preflight_receipt.json"
    assert RECEIPT.is_file(), "Revisar la salida anterior: falta el recibo del proceso."
    receipt = json.loads(RECEIPT.read_text())
    (OUTPUT / "preflight_console.log").write_bytes(CONSOLE_LOG.read_bytes())
    setup_record = OUTPUT / "isolated_runtime_receipt.json"
    setup_record.write_text(json.dumps(runtime_receipt, ensure_ascii=False, indent=2) + "\n")
else:
    # Un fallo de preparación conserva un recibo sin fingir contratos ejecutados.
    spec = importlib.util.spec_from_file_location("e3_preflight_guard_only", SCRIPT)
    preflight_api = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(preflight_api)
    receipt = {"status": "blocked_at_isolated_runtime_setup", "repository_HEAD": HEAD,
        "GPU_execution_verified": False, "new_real_data_fits": 0,
        "new_market_predictions": 0, "new_hypothesis_tests": 0,
        "environment": {"runtime_setup_error": SETUP_ERROR}, "models": [],
        "campaign_guard": preflight_api.campaign_gate()}
    OUTPUT.mkdir(parents=True)
    RECEIPT = OUTPUT / "preflight_receipt.json"
    RECEIPT.write_text(json.dumps(receipt, ensure_ascii=False, indent=2) + "\n")


## Revisar el resultado y conservar evidencia

`passed_synthetic_only` verifica el contrato artificial en el dispositivo indicado. Un bloqueo de versiones o fuentes ocurre antes de construir modelos. Una excepción de una receta queda identificada en `models` y en su `worker.log`. Los checkpoints son aleatorios y no entrenados; no son artefactos de una campaña científica.


In [ ]:
print("Estado:", receipt["status"])
print("HEAD registrado:", receipt["repository_HEAD"])
print("GPU realmente verificada:", receipt["GPU_execution_verified"])
print("Ajustes con datos reales:", receipt["new_real_data_fits"])
print("Predicciones de mercado:", receipt["new_market_predictions"])
print("Contrastes nuevos:", receipt["new_hypothesis_tests"])
print(json.dumps(receipt["environment"], ensure_ascii=False, indent=2))
for item in receipt.get("models", []):
    print(item["model"], "→", item["status"], item.get("error", ""))

# Sólo recibos y logs: no redistribuye árboles autores ni checkpoints.
EVIDENCE_ZIP = BASE / ("e3-preflight-receipts-" + RUN_ID + ".zip")
with zipfile.ZipFile(EVIDENCE_ZIP, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file() and "source_package" not in path.relative_to(OUTPUT).parts and path.suffix in (".json", ".log"):
            archive.write(path, path.relative_to(OUTPUT))
print("Recibo:", RECEIPT)
print("Paquete de evidencia:", EVIDENCE_ZIP)
try:
    from IPython.display import FileLink, display
    display(FileLink(str(EVIDENCE_ZIP)))
except ImportError:
    pass


## Guardar la ejecución completa en GitHub

Se guarda un commit con los recibos JSON, los logs y el ZIP, junto con un manifiesto de tamaños y SHA-256. El ZIP debe contener exactamente los mismos archivos de evidencia. El guardado excluye instalaciones y material temporal ajeno a ese registro, verifica los bytes remotos y crea una rama nueva `e3-colab-receipts-<RUN_ID>`.

Al completar el guardado se imprimen la URL de la ejecución y el commit. Guarda esa URL: desde ella se puede revisar la evidencia sin depender de la sesión de Colab. Un reintento con los mismos bytes reconoce el guardado existente. Si falla el permiso de escritura, corrige el secreto y ejecuta **sólo esta celda**; conserva `OUTPUT` y `EVIDENCE_ZIP` y no repitas los modelos.


In [ ]:
_E3_GITHUB_TOKEN = _e3_github_token()
try:
    GITHUB_PUBLICATION = github_api.save_run(
        OUTPUT, EVIDENCE_ZIP, _E3_GITHUB_TOKEN, HEAD, RUN_ID
    )
except Exception as error:
    if str(error) in {"github_auth_or_contents_write_permission_required", "repository_push_access_required"}:
        _E3_GITHUB_TOKEN = ""
    print("El guardado no quedó confirmado. Conserva esta sesión y revisa el error.")
    print("Si corriges GITHUB_TOKEN, reintenta sólo esta celda de guardado.")
    raise
print("Ejecución guardada en GitHub:", GITHUB_PUBLICATION["run_url"])
print("Commit:", GITHUB_PUBLICATION["commit_sha"])
(BASE / ("e3-github-publication-" + RUN_ID + ".json")).write_text(
    json.dumps(GITHUB_PUBLICATION, ensure_ascii=False, indent=2) + "\n"
)


## Guard antes de cualquier entrenamiento o evaluación real

La campaña queda bloqueada: hoy hay **0 de 3 conjuntos adicionales admitidos**. Los tres candidatos del inventario no reemplazan archivos y derechos verificados. También faltan el protocolo final con tag anterior a los nuevos ajustes, la identidad/configuración del candidato, el tratamiento y presupuesto común de los 17 sistemas y el dimensionamiento final de potencia individual y conjunta de toda la batería.

El guard valida evidencias con hashes y orden temporal del tag cuando exista un manifiesto final. Este cuaderno no contiene un lanzador de entrenamiento real. Pasar el preflight o disponer de GPU no satisface R1/R3, no convierte resultados retrospectivos en confirmatorios y no permite rebajar el estándar.


In [ ]:
guard = receipt["campaign_guard"]
print("¿Campaña real habilitada?", guard["allowed"])
for blocker in guard["blockers"]:
    print("•", blocker)
assert guard["allowed"] is False
assert receipt["new_real_data_fits"] == 0
assert receipt["new_hypothesis_tests"] == 0
print("Preflight terminado. La siguiente decisión depende de las admisiones y del protocolo.")
